In [2]:
import time

# Start the timer
start_time = time.time()

# --- operation to measure ---
total = 0
for i in range(1_000_000):
    total += i
# --- end of operation ---

end_time = time.time()
elapsed = end_time - start_time

print(f"Result:  {total}")
print(f"Elapsed: {elapsed:.4f} seconds")

Result:  499999500000
Elapsed: 0.3059 seconds


In [5]:
import time

dataset_size  = 50_000   # total training samples
batch_size    = 64
num_batches   = dataset_size // batch_size

epoch_start = time.time()

for batch_idx in range(num_batches):
    # simulate processing one batch
    time.sleep(0.01)   # replace with real training step

epoch_elapsed  = time.time() - epoch_start
samples_per_sec = dataset_size / epoch_elapsed

print(f"Epoch time:  {epoch_elapsed:.2f}s")
print(f"Throughput:  {samples_per_sec:.1f} samples/sec")

Epoch time:  7.94s
Throughput:  6299.9 samples/sec


In [8]:
import psutil
import time

def log_system_resources(label=""):
    cpu_pct  = psutil.cpu_percent(interval=1)       # average over 1 sec
    ram      = psutil.virtual_memory()
    ram_used = ram.used / (1024**3)                  # convert bytes to GB
    ram_pct  = ram.percent

    print(f"[{label}]")
    print(f"  CPU utilisation : {cpu_pct:.1f}%")
    print(f"  RAM used        : {ram_used:.2f} GB  ({ram_pct:.1f}%)")

# Use before and after each training epoch
log_system_resources("Before training")
time.sleep(5)   # simulate training
log_system_resources("After epoch 1")

[Before training]
  CPU utilisation : 2.0%
  RAM used        : 0.76 GB  (8.2%)
[After epoch 1]
  CPU utilisation : 3.0%
  RAM used        : 0.76 GB  (8.2%)


In [11]:
from torch.utils.tensorboard import SummaryWriter
import time
import random

writer = SummaryWriter(log_dir="runs/experiment_1")

for epoch in range(10):
    t0 = time.time()

    # --- simulate training step ---
    loss     = 1.0 / (epoch + 1) + random.uniform(0, 0.05)
    accuracy = 1 - loss + random.uniform(0, 0.02)
    # --------------------------------

    epoch_time = time.time() - t0

    # Log metrics to TensorBoard
    writer.add_scalar("Loss/train",    loss,        epoch)
    writer.add_scalar("Accuracy/train",accuracy,   epoch)
    writer.add_scalar("Performance/epoch_time_sec", epoch_time, epoch)
    print(f"Epoch {epoch+1}: loss={loss:.4f}, acc={accuracy:.4f}")

writer.close()

# In a separate terminal:
# tensorboard --logdir=runs
# Then open  http://localhost:6006  in a browser

Epoch 1: loss=1.0349, acc=-0.0328
Epoch 2: loss=0.5109, acc=0.5041
Epoch 3: loss=0.3764, acc=0.6401
Epoch 4: loss=0.2705, acc=0.7309
Epoch 5: loss=0.2410, acc=0.7720
Epoch 6: loss=0.1989, acc=0.8069
Epoch 7: loss=0.1524, acc=0.8621
Epoch 8: loss=0.1413, acc=0.8588
Epoch 9: loss=0.1465, acc=0.8596
Epoch 10: loss=0.1304, acc=0.8822


In [14]:
# PyTorch Automatic Mixed Precision (AMP) — minimal example
import torch
from torch.cuda.amp import GradScaler, autocast

# NOTE: MyModel, dataloader, and criterion would need to be defined
# for this code to run without NameError. This is a minimal example.

# Example placeholder for MyModel (replace with your actual model definition)
class MyModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = torch.nn.Linear(10, 1)

    def forward(self, x):
        return self.linear(x)

# Example placeholder for dataloader (replace with your actual dataloader)
dataloader = [(torch.randn(16, 10), torch.randn(16, 1)) for _ in range(5)]

# Example placeholder for criterion (replace with your actual loss function)
criterion = torch.nn.MSELoss()

# Determine if CUDA is available and set the device accordingly
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = MyModel().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# GradScaler is only needed when using autocast with CUDA
if device.type == 'cuda':
    scaler = GradScaler()          # prevents underflow in FP16 gradients
else:
    scaler = None # Or handle CPU fallback if needed, but not for AMP

for inputs, labels in dataloader:
    inputs, labels = inputs.to(device), labels.to(device)
    optimizer.zero_grad()

    if device.type == 'cuda':
        with autocast():              # forward pass in FP16
            outputs = model(inputs)
            loss    = criterion(outputs, labels)
        scaler.scale(loss).backward() # backward pass with scaled gradients
        scaler.step(optimizer)
        scaler.update()
    else:
        # Regular forward/backward pass for CPU
        outputs = model(inputs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()


Using device: cpu
